An inertial measurement unit (IMU) measures how the body it is fixed to moves. It contains a gyroscope and an accelerometer, each with three orthogonal axes, and some add a magnetometer. An IMU senses nothing outside the robot, so it never sees a wall or a landmark the way a camera or a LIDAR does. It reports the motion itself, hundreds of times a second, and the robot's pose has to be recovered by integrating those samples. This section models the two sensors and their errors, and shows that integration alone keeps the pose for only a few seconds.

A TurtleBot3 carries two IMUs once a D435i camera is mounted on it. The robot's controller board, the OpenCR 1.0, has an InvenSense ICM-20648, which replaced the MPU9250 in 2020 ([ROBOTIS](https://emanual.robotis.com/docs/en/parts/controller/opencr10/)). The D435i has its own Bosch BMI055, fixed to the camera ([librealsense](https://github.com/realsenseai/librealsense/blob/master/doc/d435i.md)). This section uses the robot's IMU. The camera's IMU appears in the [visual-inertial odometry](/aiml-common/lectures/rse/vio/index) section, where it is fused with the camera it is bolted to.

In [ ]:
%config InlineBackend.figure_formats = ["svg"]

import matplotlib.pyplot as plt
import numpy as np
from cycler import cycler
from scipy.spatial.transform import Rotation

plt.rcParams.update(
    {
        "figure.figsize": (6.5, 3.5),
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "axes.prop_cycle": cycler(
            color=["#0072B2", "#D55E00", "#009E73", "#cb77a6", "#c28600", "#1d99e0", "#7f7f7f"]
        ),
        "axes.grid": True,
        "grid.alpha": 0.3,
        "svg.fonttype": "path",
    }
)
BLUE, VERMILLION, GREEN, PINK, GOLD, SKY, GRAY = (
    "#0072B2", "#D55E00", "#009E73", "#cb77a6", "#c28600", "#1d99e0", "#7f7f7f",
)

## The gyroscope

A gyroscope measures angular velocity. A triaxial gyroscope gives the three components of the angular velocity $\boldsymbol\omega$ of the body, expressed in the body frame $\{B\}$. The sensors in phones and robots are micro-electro-mechanical systems (MEMS): a small mass vibrates at several kHz, and a rotation pushes it sideways in proportion to the rate, which is measured capacitively [@Corke2023-jx].

Integrating the angular velocity gives orientation. If $R_{WB,k}$ is the rotation from the body frame $\{B\}$ to the world frame $\{W\}$ at step $k$ and $\boldsymbol\omega_k$ is held constant over the sample interval $\delta t$, the orientation one step later is

$$
R_{WB,k+1} = R_{WB,k} \, e^{\delta t\, [\boldsymbol\omega_k]_\times},
$$

where $[\boldsymbol\omega]_\times$ is the skew-symmetric matrix of $\boldsymbol\omega$ and the matrix exponential is the rotation by angle $\|\boldsymbol\omega\|\delta t$ about $\boldsymbol\omega$. This is the [exponential map](/aiml-common/lectures/kinematics/lie-applications/index) from the kinematics chapter, applied in the body frame because that is where the gyroscope measures.

## The accelerometer

An accelerometer is a proof mass on a spring. The spring holds the mass against gravity and against the body's acceleration, so the displacement measures the difference between the two. With the world $z$ axis up and gravity $\mathbf g = (0, 0, -9.81)$ m/s², a triaxial accelerometer reports the *specific force*

$$
\mathbf f = R_{WB}^\top (\mathbf a - \mathbf g),
$$

the acceleration $\mathbf a$ of the body minus gravity, rotated into the body frame. A robot standing still on a level floor has $\mathbf a = 0$, so its accelerometer reads $(0, 0, 9.81)$: an upward acceleration of one $g$, although it is not moving. Some phone apps report the opposite sign, which adds to the confusion [@Corke2023-jx].

To get position you invert this model. Rotate the reading into the world frame with the current orientation estimate, add gravity back, and integrate twice:

$$
\mathbf a_k = R_{WB,k} \mathbf f_k + \mathbf g, \qquad
\mathbf v_{k+1} = \mathbf v_k + \mathbf a_k \delta t, \qquad
\mathbf p_{k+1} = \mathbf p_k + \mathbf v_k \delta t + \tfrac12 \mathbf a_k \delta t^2 .
$$

The orientation enters this step. An error in $R_{WB,k}$ rotates part of the 9.81 m/s² of gravity into the horizontal plane, where the integration treats it as acceleration.

The code below implements both sensor models and this integration, which is called *strapdown* integration because the sensors are strapped to the body rather than held level by gimbals. The last line checks the sign convention on a robot at rest.

In [ ]:
RATE = 200  # Hz, the TurtleBot3 IMU in the course simulator
G_W = np.array([0.0, 0.0, -9.81])  # gravity in the world frame, z up
SIGMA_G, SIGMA_A = 2e-4, 1.7e-2  # per-sample white noise of the simulated TurtleBot3 IMU


def imu_measure(R_wb, a_w, omega_b, b_g, b_a, rng, sigma_g, sigma_a):
    """One IMU sample: angular velocity and specific force, both in the body frame."""
    omega_meas = omega_b + b_g + rng.normal(0, sigma_g, 3)
    f_meas = R_wb.T @ (a_w - G_W) + b_a + rng.normal(0, sigma_a, 3)
    return omega_meas, f_meas


def strapdown(R0, p0, v0, omegas, fs, dt):
    """Integrate IMU samples into orientation, position and velocity."""
    n = len(omegas)
    Rs, ps, vs = np.empty((n + 1, 3, 3)), np.empty((n + 1, 3)), np.empty((n + 1, 3))
    Rs[0], ps[0], vs[0] = R0, p0, v0
    for k in range(n):
        a_w = Rs[k] @ fs[k] + G_W  # specific force back to acceleration
        ps[k + 1] = ps[k] + vs[k] * dt + 0.5 * a_w * dt**2
        vs[k + 1] = vs[k] + a_w * dt
        Rs[k + 1] = Rs[k] @ Rotation.from_rotvec(omegas[k] * dt).as_matrix()
    return Rs, ps, vs


rng = np.random.default_rng(0)
omega_still, f_still = imu_measure(np.eye(3), np.zeros(3), np.zeros(3), np.zeros(3), np.zeros(3), rng, 0, 0)
print("robot at rest: gyro", omega_still, " accelerometer", f_still)

## Noise and bias

A common model for any one axis of an inertial sensor is

$$
\tilde x = s\,x + b + \varepsilon,
$$

where $s$ is the scale factor, $b$ the bias and $\varepsilon$ random noise [@Corke2023-jx]. Calibration can fix the scale factor. The bias causes most of the trouble: it differs from device to device, changes with temperature and wanders slowly over time.

Integration turns each error into a growing one. White noise on the accelerometer, integrated twice, gives a position error that grows like $t^{3/2}$. A constant accelerometer bias $b$ is worse: integrated twice it gives $\tfrac12 b t^2$, so 0.05 m/s² becomes 2.5 m after 10 s and 90 m after a minute. A gyroscope bias acts through orientation. On a body that does not rotate, a bias $b_g$ about a horizontal axis tilts the orientation estimate by $b_g t$, and the tilt leaks $9.81 \sin(b_g t)$ m/s² of gravity into the horizontal acceleration. That error is integrated twice more, so the position error grows like $t^3$. A robot that turns behaves differently, as the example below shows.

## Dead reckoning a TurtleBot from its IMU

The TurtleBot below drives a counterclockwise circle of radius 1.5 m at 0.2 m/s for 60 s, a little more than one lap. Its IMU samples at 200 Hz with the white noise of the course simulator's TurtleBot3 IMU: 2e-4 rad/s on each gyro axis and 1.7e-2 m/s² on each accelerometer axis. The integration starts from the true pose and velocity, so every error that appears comes from the sensor.

Four versions of the same drive differ only in the bias. The biases are illustrative, of the size the course simulator gives the D435i's IMU: 3, -4 and 2 mrad/s on the gyro axes and 0.04, -0.03 and 0.05 m/s² on the accelerometer axes.

In [ ]:
def turtlebot_loop(radius=1.5, speed=0.2, duration=60.0, rate=RATE):
    """True motion of a TurtleBot driving a counterclockwise circle at constant speed."""
    w = speed / radius
    t = np.arange(int(duration * rate) + 1) / rate
    th = np.pi / 2 + w * t
    p = np.stack([radius * np.cos(w * t), radius * np.sin(w * t), np.zeros_like(t)], 1)
    v = np.stack([speed * np.cos(th), speed * np.sin(th), np.zeros_like(t)], 1)
    a = np.stack([-speed * w * np.sin(th), speed * w * np.cos(th), np.zeros_like(t)], 1)
    R = Rotation.from_euler("z", th[:, None]).as_matrix()
    om = np.tile([0.0, 0.0, w], (len(t), 1))
    return t, R, p, v, a, om


def measure_all(R_true, a_w, om_b, b_g, b_a, rng, sigma_g, sigma_a):
    """One IMU sample at the start of every interval."""
    out = [imu_measure(R, a, w, b_g, b_a, rng, sigma_g, sigma_a)
           for R, a, w in zip(R_true[:-1], a_w[:-1], om_b[:-1])]
    return np.array([o[0] for o in out]), np.array([o[1] for o in out])


B_G = np.array([3e-3, -4e-3, 2e-3])  # rad/s, a constant gyro bias
B_A = np.array([0.04, -0.03, 0.05])  # m/s^2, a constant accelerometer bias

t, R_true, p_true, v_true, a_true, om_true = turtlebot_loop()
cases = {
    "no bias": (0 * B_G, 0 * B_A),
    "gyro and accelerometer bias": (B_G, B_A),
    "gyro bias only": (B_G, 0 * B_A),
    "accelerometer bias only": (0 * B_G, B_A),
}
paths, errors = {}, {}
for name, (b_g, b_a) in cases.items():
    om, f = measure_all(R_true, a_true, om_true, b_g, b_a, rng, SIGMA_G, SIGMA_A)
    Rs, ps, _ = strapdown(R_true[0], p_true[0], v_true[0], om, f, 1 / RATE)
    paths[name], errors[name] = ps, np.linalg.norm(ps - p_true, axis=1)
    if name == "gyro bias only":
        # tilt: the angle between the estimated and the true body z axis
        tilt = np.degrees(np.arccos(np.clip(np.sum(Rs[:, :, 2] * R_true[:, :, 2], axis=1), -1, 1)))
    print(f"{name:>28}: position error after 1, 10, 60 s =",
          [round(float(errors[name][int(s * RATE)]), 2) for s in (1, 10, 60)], "m")
print("tilt from the gyro bias after 10, 30, 60 s:", [round(float(tilt[int(s * RATE)]), 1) for s in (10, 30, 60)], "deg")

For comparison, wheel odometry integrates the commanded wheel speeds of the same loop. The wheels slip by the amounts used in the [EKF localization](/aiml-common/lectures/rse/ekf-localization/index) section, a standard deviation of 5% of each wheel speed plus 0.1 rad/s.

In [ ]:
r, d = 0.033, 0.1435  # TurtleBot3 Waffle wheel radius and half wheel separation, in meters


def wrap(angle):
    """Wrap an angle to [-pi, pi)."""
    return (angle + np.pi) % (2 * np.pi) - np.pi


def twist(a):
    """Forward speed v and turning rate w from the wheel speeds a = (u_L, u_R)."""
    u_L, u_R = a
    return r / 2 * (u_L + u_R), r / (2 * d) * (u_R - u_L)


def g(a, s, dt):
    """Velocity motion model: the pose after holding the wheel speeds a for dt."""
    v, w = twist(a)
    x, y, th = s
    if abs(w) < 1e-9:
        return np.array([x + v * dt * np.cos(th), y + v * dt * np.sin(th), th])
    return np.array([
        x - v / w * np.sin(th) + v / w * np.sin(th + w * dt),
        y + v / w * np.cos(th) - v / w * np.cos(th + w * dt),
        wrap(th + w * dt),
    ])


# the same loop driven on wheels whose speeds are perturbed by noise every 0.5 s
odo_rng, dt_odo = np.random.default_rng(1), 0.5
w_loop = 0.2 / 1.5
a_cmd = np.array([(0.2 - d * w_loop) / r, (0.2 + d * w_loop) / r])
sigma_u = 0.05 * np.abs(a_cmd) + 0.1
s_true = s_odo = np.array([1.5, 0.0, np.pi / 2])
odo_err = [0.0]
for _ in range(int(60 / dt_odo)):
    s_true = g(a_cmd + odo_rng.normal(0, sigma_u), s_true, dt_odo)
    s_odo = g(a_cmd, s_odo, dt_odo)
    odo_err.append(np.hypot(*(s_odo[:2] - s_true[:2])))
odo_err = np.array(odo_err)
print(f"wheel odometry: position error after 60 s = {odo_err[-1]:.2f} m")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.6))
colors = [BLUE, VERMILLION, PINK, GOLD]
for (name, e), c in zip(errors.items(), colors):
    ax.semilogy(t[1:], e[1:], color=c, label=f"IMU, {name}")
ax.semilogy(np.arange(len(odo_err))[1:] * dt_odo, odo_err[1:], color=GREEN, ls="--", label="wheel odometry")
ax.set_xlabel("time (s)")
ax.set_ylabel("position error (m)")
ax.set_xlim(0, 60)
ax.set_ylim(1e-3, 3e3)
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

The vertical axis is logarithmic. Without bias the IMU estimate stays within 2 cm of the true position for the first 10 s and then drifts away, reaching 2.0 m after 60 s. Wheel odometry with slipping wheels ends 0.48 m off, so on a ground robot even a bias-free IMU loses to the wheels within a minute.

With bias the estimate is 7.8 m off after 10 s and 643 m after 60 s. The two single-bias curves show where the error comes from. The accelerometer bias alone gives 3.5 m after 10 s, close to the $\tfrac12 b t^2$ estimate. The gyroscope bias alone gives almost all of the final error. Because the robot keeps turning, the horizontal part of the bias points in a different world direction as the lap goes on, so the tilt does not grow without bound: it is 2.7° after 10 s, 3.8° after 30 s and 3.4° after 60 s, rising and falling below 5° as the robot turns. Even 3° leaks about 0.5 m/s² of gravity into the horizontal acceleration, and that integrated twice gives the hundreds of meters in the plot.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(p_true[:, 0], p_true[:, 1], color=GREEN, lw=2, label="true path")
ax.plot(*paths["no bias"][:, :2].T, color=BLUE, label="IMU, no bias")
k10 = 10 * RATE
ax.plot(*paths["gyro and accelerometer bias"][:k10, :2].T, color=VERMILLION, label="IMU with bias, first 10 s")
ax.plot(*paths["gyro and accelerometer bias"][k10, :2], "o", color=VERMILLION, ms=5)
ax.plot(*p_true[0, :2], "s", color="black", ms=5, label="start")
ax.set_aspect("equal")
ax.set_xlabel("$x$ (m)")
ax.set_ylabel("$y$ (m)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)
fig.tight_layout()
plt.show()

The path plot shows the same thing in space. The bias-free estimate follows the circle for most of the lap and then drifts outward. The biased estimate leaves the circle within seconds; the dot marks where it is after 10 s.

An IMU alone therefore keeps the pose for only a few seconds. It is still worth carrying: it samples hundreds of times a second and needs nothing from the environment, so it keeps working in the dark or in front of a blank wall. Navigation systems pair it with a sensor that measures the world, such as a camera, and estimate the biases online. The [visual-inertial odometry](/aiml-common/lectures/rse/vio/index) section builds that filter.

The TurtleBot3 IMU in the course simulator has white noise but no bias, so dead reckoning in Gazebo drifts like the blue curve above, far less than a real robot does. The simulated D435i IMU has both noise and a drifting bias.

**Key references**: [@Corke2023-jx]